In [1]:
%load_ext autoreload
%autoreload 2

from src import loaders

In [9]:
mc_sl_2x2 = "tests/snake_ladder/mc_2x2.pm"
mon_sl_2x2 = "tests/snake_ladder/mon_2x2.pm"

mc_sl_2x2_deadlock = "tests/snake_ladder/mc_2x2_deadlock.pm"
mon_sl_2x2_deadlock = "tests/snake_ladder/mon_2x2_deadlock.pm"

mc_sl_nxn = "tests/snake_ladder/mc_nxn.pm"
mon_sl_nxn = "tests/snake_ladder/mon_nxn.pm"

mc_simple = "tests/simple/mc1.pm"
mon_simple = "tests/simple/mon1.pm"

gb_r = "tests/gb_reachability.pm"
gb_e = "tests/gb_exact.pm"

mc, n, ladders, snakes = loaders.load_mc(mc_sl_nxn)
mon = loaders.load_dfa(mon_sl_nxn)
gb = loaders.load_dfa(gb_e)

In [10]:
from stormvogel.show import show
# show(gb)
# show(mon)
show(mc)
print(mc)

Output()

ModelType.DTMC with name None

States:
State 0 with labels ['init', '[pos=0]'] and features {}
State 1 with labels ['ladder', '[pos=1]'] and features {}
State 2 with labels ['normal', '[pos=2]'] and features {}
State 3 with labels ['ladder', '[pos=3]'] and features {}
State 4 with labels ['normal', '[pos=4]'] and features {}
State 5 with labels ['normal', '[pos=9]'] and features {}
State 6 with labels ['normal', '[pos=5]'] and features {}
State 7 with labels ['normal', '[pos=6]'] and features {}
State 8 with labels ['ladder', '[pos=11]'] and features {}
State 9 with labels ['snake', '[pos=7]'] and features {}
State 10 with labels ['normal', '[pos=8]'] and features {}
State 11 with labels ['snake', '[pos=10]'] and features {}
State 12 with labels ['snake', '[pos=12]'] and features {}
State 13 with labels ['normal', '[pos=13]'] and features {}
State 14 with labels ['normal', '[pos=14]'] and features {}
State 15 with labels ['normal', '[pos=15]'] and features {}
State 16 with labels ['goo

In [15]:
from src.MDP_product import MC_MON_Product

pomdp = MC_MON_Product(mc, mon, gb)
pomdp.apply_spec('P>=0.5 [ F<=0 "good"]')
pomdp.create_product()
# pomdp.add_ret_to_bmecs()
# pomdp.remove_unreachable_states()

# show(pomdp.pomdp)

induced_mc, poss = pomdp.check_paynt_prop('Pmax=? [ (F "stop") ]')

16 True
--------------------
Synthesis summary:
optimality objective: Pmax=? [F "stop"] 

method: AR, synthesis time: 0.01 s
number of holes: 13, family size: 1e7, quotient: 444 states / 1600 actions
explored: 100 %
MDP stats: avg MDP size: 26, iterations: 42

optimum: 0.748759
--------------------
counterexample found:  A(0,0)=ladder, A(1,0)=normal, A(2,0)=ladder, A(3,0)=normal, A(4,0)=normal, A(5,0)=end, A(6,0)=ladder, A(7,0)=ladder, A(8,0)=ladder, A(9,0)=ladder, A(10,0)=ladder, A(11,0)=ladder, A(12,0)=ladder 
--------------------------------------

s0, labels=init s0 l0 g0 [pos=0]
	--[0, 0:ladder]-->
s37, labels=[pos=1] g0 l1 ladder s1
	--[1, 1:normal]-->
s75, labels=[pos=9] g0 l2 normal s5
	--[2, 0:ladder]-->
s112, labels=[pos=11] g0 l3 ladder s8
	--[3, 1:normal]-->
s152, labels=[pos=14] g0 l4 normal s14
	--[4, 1:normal]-->
s205, labels=[pos=16] accepting g1 good l5 normal s16
	--[5, 3:end]-->
s1, labels=goal
it took 3 tries until the goal was reached


In [16]:
%matplotlib notebook
from IPython.display import HTML
from src.draw import animate_player_movement
import math

goal_squares = []
for state in pomdp.mc.states.values():
    if "good" in state.labels:
        goal_squares.append(state.id)
print(goal_squares)

player_path = poss

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

[16]


<IPython.core.display.Javascript object>

In [ ]:
from stormvogel.mapping import stormpy_to_stormvogel
from stormvogel.show import show
from stormpy import model_checking, parse_properties

imc =  stormpy_to_stormvogel(induced_mc)
result_goal = model_checking(induced_mc, parse_properties('Pmax=? [F "goal"]')[0])
result_stop = model_checking(induced_mc, parse_properties('Pmax=? [F "stop"]')[0])
prop_goal = result_goal.at(induced_mc.initial_states[0])
prop_stop = result_stop.at(induced_mc.initial_states[0])
print(f"probability to reach goal={prop_goal} and stop={prop_stop}. Total={prop_goal+prop_stop}")
show(imc)


In [ ]:
# scheduler = pomdp.check_storm_prop('Pmax=? [ F "goal" ]')